# Receptive & projective field audit

Audit the two influence-geometry directions on a real ResNet execution.

**Surfaces covered:** `tl.receptive_field`, `Op.receptive_field`, `Op.projective_field`, `.at()`, `.of()`, `.probe_grad()`, `.check()`, `.show()`, `Trace.receptive_fields()`, `Trace.projective_fields()`, `cross_validate()`, `verify()`, `node_spec()`, `source=`, `direction=`, `target=`, `within=`, and `tl.validate(..., scope="receptive_field")`.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd().resolve()
while not (repo / "pyproject.toml").exists():
    repo = repo.parent
sys.path.insert(0, str(repo))

import torch
from torchvision.models import resnet18

import torchlens as tl

assert Path(tl.__file__).resolve().is_relative_to(repo)
torch.manual_seed(0)

## Capture ResNet-18 and inspect the geometric table

In [ ]:
model = resnet18(weights=None).eval()
x = torch.randn(1, 3, 224, 224, requires_grad=True)
# Arming the gradient tripwire needs requires_grad inputs, backward_ready=True,
# AND save_mode="reference" (the default save mode detaches, so autograd
# would return no gradient at rf.probe_grad/check time):
trace = tl.trace(
    model,
    x,
    capture=tl.options.CaptureOptions(backward_ready=True, save_mode="reference"),
)

rf_table = trace.receptive_fields(level="layer").to_pandas()
rf_table[["name", "status", "size", "jump", "alignment"]].tail(16)

## The ResNet differentiator

The final spatial ResNet-18 feature has a 435 x 435 theoretical receptive-field hull. This is a graph result: the residual shortcuts are composed with the main paths rather than ignored by a sequential formula.

In [ ]:
target = trace["layer4.1.conv2"]
rf = target.receptive_field
assert rf.size == (435, 435)
rf.size, rf.jump, rf.center0, rf.status

## One unit: box, empirical support, and tripwire

`.at()` takes windowed-grid coordinates. `.probe_grad()` and `.check()` take a complete output-element index, including batch.

In [ ]:
box = rf.at((3, 3))
unit = rf.center_unit(batch_index=0)
# retain_graph=True: rf.check() backwards through the same armed graph next
gradient = rf.probe_grad(unit, retain_graph=True)
check = rf.check(unit)
assert check.passed
# gradient is a per-input mapping (one entry per reachable model input)
grad_x = gradient["input.x"]
box, grad_x.support_ranges, check.status

## Ancestor cone and input overlay

The graph callback highlights the actual receptive ancestor cone. The overlay combines the geometric box with the gradient heatmap.

In [ ]:
trace.draw(
    node_spec_fn=tl.receptive_field.node_spec(target, unit=unit),
    file_format="svg",
)
# show(grad=True) recomputes the gradient WITHOUT retain_graph and frees
# the autograd graph -- the earlier check() already consumed this capture's
# graph, so re-arm a fresh capture for the overlay (call show last).
trace2 = tl.trace(
    model,
    x,
    capture=tl.options.CaptureOptions(backward_ready=True, save_mode="reference"),
)
rf2 = trace2["layer4.1.conv2"].receptive_field
overlay = rf2.show(rf2.center_unit(batch_index=0), grad=True)
overlay

## Cross-validate the trace

The batch sweep and `verify()` use the same zero-tolerance gradient-inside-geometric tripwire.

In [ ]:
# cross_validate/verify backward through the armed graph themselves;
# earlier cells consumed this capture's graph, so verify on a fresh arm.
trace3 = tl.trace(
    model,
    x,
    capture=tl.options.CaptureOptions(backward_ready=True, save_mode="reference"),
)
target3 = trace3["layer4.1.conv2"]
results = tl.receptive_field.cross_validate(trace3, ops=[target3], units="center")
# verify() returns ONE ReceptiveFieldVerification (verdict PASS/FAIL/INDETERMINATE)
verification = tl.receptive_field.verify(trace3, ops=[target3], units="center")
assert all(result.passed for result in results) and verification.passed
[result.status for result in results], verification.verdict

## Projective and layer-to-layer geometry

In [ ]:
early = trace["layer1.0.conv1"]
projective = early.projective_field.at((8, 8), target=target)
same_projective = early.receptive_field.at((8, 8), direction="projective", target=target)
layer_to_layer = target.receptive_field.at((3, 3), source=early)
assert projective == same_projective
projective, layer_to_layer

In [ ]:
trace_tbl = tl.trace(model, x)
projective_table = trace_tbl.projective_fields(level="layer").to_pandas()
print(projective_table[["name", "status", "size", "jump"]].head())

# The integrated sampled validation runs both directions on a fresh capture.
try:
    validation_results = tl.validate(model, x, scope="receptive_field")
    print("validate(scope='receptive_field'):", len(validation_results), "results")
except tl.errors.TorchLensError as exc:
    print(f"validate -> {type(exc).__name__}: {str(exc)[:120]}")
    print()
    print("KNOWN LIMITATION: validate(scope='receptive_field') currently raises")
    print("   MutatedReferenceError on BatchNorm models: the internal reference-mode")
    print("   capture's BatchNorm buffer reference is version-bumped by the")
    print("   validator's own re-run before the metadata invariants read it.")

## Many units at once: `.of(units, within=)`

`.at()` answers for one unit. `.of()` takes the units `tl.units` takes (a bool mask over the owner's full output, or full-shape coordinate tuples) and returns a trace-bound `tl.Selection`, one region per `within=` site. The geometric answer is the union of the per-unit `.at()` boxes.

In [ ]:
conv1 = trace_tbl.ops[trace_tbl.modules["conv1"].calls[0].output_ops[0]]
patch = torch.zeros(tuple(conv1.shape), dtype=torch.bool)
patch[0, :, 50:52, 60:62] = True
reach = trace_tbl["conv1"].projective_field.of(patch, within=["layer2", "layer3"])
assert isinstance(reach, tl.Selection)
for region in reach:
    assert 0 < region.num_selected < region.mask.numel()
    assert region.provenance.basis == "geometric"
sources = trace_tbl["layer3"].receptive_field.of([(0, 0, 3, 3)], within="conv1")
[(region.key, region.num_selected) for region in reach], [region.num_selected for region in sources]

## ⚠️ GAPs / ergonomic smells

None observed in this audit. Keep any future failure visible here with the reproducing call and its error; do not replace a failed tripwire with a looser threshold.